In [1]:
import sys
sys.path.append('../')
import os
os.environ['PATH'] = os.environ['PATH'] + os.pathsep + '/home/ubuntu/miniconda3/envs/utrtool/bin/'

import pybedtools
import HTSeq

from core import utr_reference as utr_ref
from core import protocols

In [2]:
gtf_path = "../tests/test_data/test_mus_musculus_GRCm38_102.gtf"
%time utr = utr_ref.extract_feature(gtf_path, 'three_prime_utr')
print(utr)

2021-07-12 18:06:18,765 - INFO - Parsing GTF reference file to retrieve only the UTR annotations
INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'transcript_id', 'transcript_version', 'exon_number', 'gene_name', 'gene_source', 'gene_biotype', 'havana_gene', 'havana_gene_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'protein_id', 'protein_version', 'tag', 'transcript_support_level', 'ccds_id', 'exon_id', 'exon_version', 'havana_transcript', 'havana_transcript_version']
2021-07-12 18:06:18,851 - INFO - Finished parsing GTF file
CPU times: user 76.3 ms, sys: 12.2 ms, total: 88.5 ms
Wall time: 86.6 ms
     seqname          source          feature      start        end  score  \
4          1         ensembl  three_prime_utr  130697017  130697884    NaN   
5          1         ensembl  three_prime_utr  170885460  170885881    NaN   
6          1         ensembl  three_prime_utr   77367185   77369671    NaN   
21         1  ensembl_havana  three_prime

In [3]:
feature_list = utr["feature"].unique()
print(feature_list)

['three_prime_utr']


In [4]:
utr['seqname'].unique()

array(['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19',
       '2', '3', '4', '5', '6', '7', '8', '9', 'X', 'Y'], dtype=object)

In [5]:
utr = utr.sort_values(by=['seqname', 'start'])  # sort by chr and start for bed merge
utr.fillna(value={'score': 0}, inplace=True)
print(utr)

     seqname          source          feature      start        end  score  \
100        1          havana  three_prime_utr    7160892    7161063    0.0   
107        1          havana  three_prime_utr    9552943    9552976    0.0   
108        1          havana  three_prime_utr    9553318    9553472    0.0   
21         1  ensembl_havana  three_prime_utr   10024601   10024978    0.0   
70         1          havana  three_prime_utr   15843083   15843470    0.0   
...      ...             ...              ...        ...        ...    ...   
1980       X          havana  three_prime_utr  166189501  166189820    0.0   
1981       X          havana  three_prime_utr  167476787  167478076    0.0   
1982       X          havana  three_prime_utr  167497813  167497949    0.0   
1950       X  ensembl_havana  three_prime_utr  169776378  169776476    0.0   
1999       Y          havana  three_prime_utr    1224497    1224546    0.0   

     strand frame             gene_id gene_version  ...  \
100 

In [6]:
annotations = ['gene_id', 'score', 'strand', 'transcript_id', 'gene_name']
columns, ann_idx = utr_ref.prepare_annotations_index(annotations)
utr_bed = pybedtools.BedTool.from_dataframe(utr, columns=columns)

utr_reduced = utr_ref.reduce_regions(utr_bed,
annotations_index=ann_idx)  # find and merge overlapping regions


2021-07-12 18:06:19,019 - INFO - Merging overlapping UTR regions


In [7]:
print(utr)

     seqname          source          feature      start        end  score  \
100        1          havana  three_prime_utr    7160892    7161063    0.0   
107        1          havana  three_prime_utr    9552943    9552976    0.0   
108        1          havana  three_prime_utr    9553318    9553472    0.0   
21         1  ensembl_havana  three_prime_utr   10024601   10024978    0.0   
70         1          havana  three_prime_utr   15843083   15843470    0.0   
...      ...             ...              ...        ...        ...    ...   
1980       X          havana  three_prime_utr  166189501  166189820    0.0   
1981       X          havana  three_prime_utr  167476787  167478076    0.0   
1982       X          havana  three_prime_utr  167497813  167497949    0.0   
1950       X  ensembl_havana  three_prime_utr  169776378  169776476    0.0   
1999       Y          havana  three_prime_utr    1224497    1224546    0.0   

     strand frame             gene_id gene_version  ...  \
100 

In [8]:
print(utr_reduced.head())

1	7160892	7161063	ENSMUSG00000051285	0.0	+	ENSMUST00000182675	Pcmtd1
 1	9552943	9552976	ENSMUSG00000025911	0.0	+	ENSMUST00000130927	Adhfe1
 1	9553318	9553472	ENSMUSG00000025911	0.0	+	ENSMUST00000130927	Adhfe1
 1	10024601	10024978	ENSMUSG00000025917	0.0	-	ENSMUST00000027050	Cops5
 1	15843083	15843470	ENSMUSG00000025925	0.0	+	ENSMUST00000027057	Terf1
 1	30944938	30945030	ENSMUSG00000117310	0.0	-	ENSMUST00000233506	Ptp4a1
 1	33734702	33734858	ENSMUSG00000004768	0.0	+	ENSMUST00000138024	Rab23
 1	34552334	34552678	ENSMUSG00000037529	0.0	-	ENSMUST00000190790	Prss40
 1	34811830	34812739	ENSMUSG00000037509	0.0	+	ENSMUST00000159747	Arhgef4
 1	36428102	36428216	ENSMUSG00000001143	0.0	-	ENSMUST00000001171	Lman2l
 None


In [9]:
utr_reduced = utr_ref.generate_utr_name(utr_reduced)
utr_reduced.head()

,chrom,start,end,name,score,strand,thickStart,thickEnd
utr_name,,,,,,,,
ENSMUSG00000051285,1,7160892,7161063,ENSMUSG00000051285,0.0,+,ENSMUST00000182675,Pcmtd1
ENSMUSG00000025911,1,9552943,9552976,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025911_2,1,9553318,9553472,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025917,1,10024601,10024978,ENSMUSG00000025917,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925,1,15843083,15843470,ENSMUSG00000025925,0.0,+,ENSMUST00000027057,Terf1


In [10]:
len(utr_reduced.index.values)

1016

In [11]:
utr_reduced = utr_ref.correct_column_names(utr_reduced)
utr_reduced.head()

,chrom,start,end,name,score,strand,transcript_id,gene_name
utr_name,,,,,,,,
ENSMUSG00000051285,1,7160892,7161063,ENSMUSG00000051285,0.0,+,ENSMUST00000182675,Pcmtd1
ENSMUSG00000025911,1,9552943,9552976,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025911_2,1,9553318,9553472,ENSMUSG00000025911,0.0,+,ENSMUST00000130927,Adhfe1
ENSMUSG00000025917,1,10024601,10024978,ENSMUSG00000025917,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925,1,15843083,15843470,ENSMUSG00000025925,0.0,+,ENSMUST00000027057,Terf1


In [12]:
from utrcalling import utr_mapping

In [13]:
intervals = utr_mapping.generate_utr_intervals(utr_reduced)
ct = 0
for name, interval in intervals.items(): 
    print(name, interval, interval.start_d, sep=': ')
    ct += 1
    if ct == 5:
        break

Generating genomic interval for UTR number: 1016it [00:00, 278232.75it/s]ENSMUSG00000051285: 1:[7160891,7161063)/+: 7160891
ENSMUSG00000025911: 1:[9552942,9552976)/+: 9552942
ENSMUSG00000025911_2: 1:[9553317,9553472)/+: 9553317
ENSMUSG00000025917: 1:[10024600,10024978)/-: 10024977
ENSMUSG00000025925: 1:[15843082,15843470)/+: 15843082



In [14]:
print(intervals['ENSMUSG00000005804'], intervals['ENSMUSG00000025917'], sep='\n')

2:[122746630,122749475)/+
1:[10024600,10024978)/-


In [15]:
alignments = ["../tests/test_data/test_UTRoldNSCgenome_bwt2_102_sort.bam", "../tests/test_data/test_UTRyoungNSCgenome_bwt2_102_sort.bam"]
protocol = protocols.Protocol(platform='10x', version=2)
read_mappings = [utr_mapping.map_reads_to_utr(alignment, intervals, protocol, read='first') for alignment in alignments]

2021-07-12 18:06:19,324 - INFO - Protocol args:

2021-07-12 18:06:19,325 - INFO - Function to get read information: <function Protocol.get_reads_identity_tenx_2 at 0x7f35f7462280>

2021-07-12 18:06:19,326 - INFO - Function to check if strand matches: <function Protocol.check_strand_matches_10x at 0x7f35f7462430>

2021-07-12 18:06:19,326 - INFO - Position counts merging method: intersection

2021-07-12 18:06:19,327 - INFO - Extracting reads from ../tests/test_data/test_UTRoldNSCgenome_bwt2_102_sort.bam and mapping them to the reference UTR
Mapping reads to reference UTR number: 100%|██████████| 1016/1016 [00:00<00:00, 5100.01it/s]
2021-07-12 18:06:19,541 - DEBUG - Size of mapped reads object in bytes: 290408        

Memory usage:20.2
2021-07-12 18:06:19,542 - INFO - Extracting reads from ../tests/test_data/test_UTRyoungNSCgenome_bwt2_102_sort.bam and mapping them to the reference UTR
Mapping reads to reference UTR number: 100%|██████████| 1016/1016 [00:00<00:00, 4749.61it/s]
2021-07-12

In [16]:
print(read_mappings[0].ENSMUSG00000005804)

MappedReadInformation(utr_name='ENSMUSG00000005804', read_start_positions=[122748098, 122748087, 122748096], read_strands=['-', '-', '-'], read_barcodes=['GATGAAACATTAACCG', 'TGCGCAGAGTCCATAC', 'GCTGGGTGTACGCACC'], read_umis=['CAACACAAG', 'ATCGAGAGT', 'CTGAGTATG'], read_lengths=[51, 37, 44], utr_sizes_from_reads=[1468, 1457, 1466])


In [17]:
bam_file = HTSeq.BAM_Reader(alignments[0])
for alignment in bam_file[intervals['ENSMUSG00000005804']]:
    print(alignment.iv, alignment.pe_which)

2:[122747783,122747885)/+ second
2:[122747812,122747913)/+ second
2:[122747857,122747958)/+ second
2:[122748048,122748099)/- first
2:[122748052,122748089)/- first
2:[122748053,122748097)/- first


In [18]:
print(read_mappings[0].ENSMUSG00000025917)

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024879, 10024880, 10024882, 10024888], read_strands=['+', '+', '+', '+'], read_barcodes=['CGTTGGGCAGTCAGAG', 'CCGTGGAGTCTCGTTC', 'GTAGTCAAGCTATGCT', 'TGCGCAGAGGCATGTG'], read_umis=['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT'], read_lengths=[46, 50, 48, 46], utr_sizes_from_reads=[98, 97, 95, 89])


In [19]:
print(intervals['ENSMUSG00000025917'])
for alignment in bam_file[intervals['ENSMUSG00000025917']]:
    print(alignment.iv, alignment.pe_which, alignment.iv.start_d, protocol.check_strand_matches(alignment.iv.strand, intervals['ENSMUSG00000025917'].strand), (alignment.iv.start_d >= interval.start_d and alignment.iv.start_d <= interval.end_d))


1:[10024600,10024978)/-
1:[10024824,10024870)/+ first 10024824 True False
1:[10024829,10024879)/+ first 10024829 True False
1:[10024829,10024877)/+ first 10024829 True False
1:[10024833,10024879)/+ first 10024833 True False
1:[10024887,10024988)/- second 10024987 False False
1:[10024907,10025008)/- second 10025007 False False
1:[10024951,10025050)/- second 10025049 False False
1:[10024958,10025059)/- second 10025058 False False
1:[10024969,10025066)/- second 10025065 False False
1:[10024970,10025066)/- second 10025065 False False


In [20]:
dct = read_mappings[0].todict()
for key, value in dct.items():
    print(key, value, sep='\n')
    break


ENSMUSG00000025917
{'utr_name': 'ENSMUSG00000025917', 'read_start_positions': [10024879, 10024880, 10024882, 10024888], 'read_strands': ['+', '+', '+', '+'], 'read_barcodes': ['CGTTGGGCAGTCAGAG', 'CCGTGGAGTCTCGTTC', 'GTAGTCAAGCTATGCT', 'TGCGCAGAGGCATGTG'], 'read_umis': ['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT'], 'read_lengths': [46, 50, 48, 46], 'utr_sizes_from_reads': [98, 97, 95, 89]}


In [21]:
read_mappings[1]['ENSMUSG00000005804', 'ENSMUSG00000025917']

[MappedReadInformation(utr_name='ENSMUSG00000005804', read_start_positions=[122748094, 122748093], read_strands=['-', '-'], read_barcodes=['AGACGTTAGTTGCAGG', 'GGCGACTGTAGCAAAT'], read_umis=['CTCATGGTA', 'ACCGGGTCC'], read_lengths=[44, 43], utr_sizes_from_reads=[1464, 1463]),
 MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024660, 10024882, 10024876, 10024880, 10024893, 10024899], read_strands=['+', '+', '+', '+', '+', '+'], read_barcodes=['CGACCTTTCTGCTGCT', 'TGTTCCGAGGGTCGAT', 'GTACGTAAGCGTTTAC', 'GTGCGGTAGGACAGCT', 'GCTGCAGCACTGTGTA', 'GTCCTCAGTAAACGCG'], read_umis=['TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], read_lengths=[43, 45, 54, 50, 41, 41], utr_sizes_from_reads=[317, 95, 101, 97, 84, 78])]

In [22]:
read_mappings[0][0:4:2]

['ENSMUSG00000025917', 'ENSMUSG00000037509']

In [23]:
from utrcalling import peak_calling

/home/ubuntu/miniconda3/envs/utrtool/lib/python3.9/site-packages/anndata/core/anndata.py:17: FutureWarning: pandas.core.index is deprecated and will be removed in a future version.  The public classes are available in the top-level namespace.
  from pandas.core.index import RangeIndex


In [24]:
positions_counts_merged = peak_calling.merge_samples_utr_sizes(read_mappings)
print(len(positions_counts_merged), positions_counts_merged.ENSMUSG00000025917.todict(), sep='\n\n')

2021-07-12 18:06:20,487 - INFO - Merging all samples' UTR sizes
100%|██████████| 2/2 [00:00<00:00, 77.80it/s]66

{'utr_name': 'ENSMUSG00000025917', 'read_start_positions': [10024879, 10024880, 10024882, 10024888, 10024660, 10024882, 10024876, 10024880, 10024893, 10024899], 'read_strands': ['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], 'read_barcodes': ['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], 'read_umis': ['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT', 'TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], 'read_lengths': [46, 50, 48, 46, 43, 45, 54, 50, 41, 41], 'utr_sizes_from_reads': [98, 97, 95, 89, 317, 95, 101, 97, 84, 78]}



In [25]:
utr_subsetted = peak_calling.subset_reference_data(positions_counts_merged, utr_reduced)
print(f'sizes: subset: {len(utr_subsetted)}, original: {len(utr_reduced)}')


2021-07-12 18:06:20,543 - INFO - Subsetting UTR reference data
sizes: subset: 66, original: 1016


In [26]:
utr_subsetted.iloc[0]

chrom                             1
start                      10024601
end                        10024978
name             ENSMUSG00000025917
score                           0.0
strand                            -
transcript_id    ENSMUST00000027050
gene_name                     Cops5
Name: ENSMUSG00000025917, dtype: object

In [27]:
utr_subsetted.head()

,chrom,start,end,name,score,strand,transcript_id,gene_name
utr_name,,,,,,,,
ENSMUSG00000025917,1,10024601,10024978,ENSMUSG00000025917,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925,1,15843083,15843470,ENSMUSG00000025925,0.0,+,ENSMUST00000027057,Terf1
ENSMUSG00000037509,1,34811830,34812739,ENSMUSG00000037509,0.0,+,ENSMUST00000159747,Arhgef4
ENSMUSG00000026356,1,128412118,128412347,ENSMUSG00000026356,0.0,-,ENSMUST00000064309,Dars
ENSMUSG00000058267,1,160199611,160199812,ENSMUSG00000058267,0.0,+,ENSMUST00000097193,Mrps14


In [28]:
%time ft = protocol.filter_utr_sizes(positions_counts_merged)

2021-07-12 18:06:20,628 - INFO - Filtering samples UTR sizes
CPU times: user 11.3 ms, sys: 0 ns, total: 11.3 ms
Wall time: 10.7 ms


In [29]:
import pandas as pd
vivo_anno_path = "../tests/test_data/vivo_anno.csv"
vivo_anno = pd.read_csv(vivo_anno_path)

In [30]:
vivo_anno.head()

,CellBarcode,CellType,CellAge,pTime
0,AAACCTGCACCAGTTA-1,qNSC1,old,4.018431
1,AAACCTGCATGGTCAT-1,qNSC2,old,14.861303
2,AAACGGGCACCGGAAA-1,qNSC2,old,20.304435
3,AAACGGGCAGCAGTTT-1,qNSC2,old,10.686136
4,AAACGGGGTAAGTTCC-1,qNSC2,old,10.351203


In [31]:
len(ft)

66

In [32]:
ft.ENSMUSG00000025917

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024879, 10024880, 10024882, 10024888, 10024660, 10024882, 10024876, 10024880, 10024893, 10024899], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_barcodes=['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], read_umis=['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT', 'TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], read_lengths=[46, 50, 48, 46, 43, 45, 54, 50, 41, 41], utr_sizes_from_reads=[98, 97, 95, 89, 317, 95, 101, 97, 84, 78])

In [33]:
counter = 0
control = 0
barcodes_real = vivo_anno.CellBarcode
for barcodes in ft.read_barcodes:
    control +=1
    if set(barcodes).issubset(barcodes_real):
        counter += 1
print(counter, control, sep='\n\n')

31

66


In [34]:
%time filts = peak_calling.filter_barcodes(ft, vivo_anno_path)
print('\n\n')
len(filts)

2021-07-12 18:06:20,813 - INFO - Filtering cell barcodes
CPU times: user 114 ms, sys: 0 ns, total: 114 ms
Wall time: 113 ms





66

In [35]:
counter = 0
control = 0
barcodes_real = vivo_anno.CellBarcode
for barcodes in filts.read_barcodes:
    control +=1
    if set(barcodes).issubset(barcodes_real):
        counter += 1
print(counter, control, sep='\n\n')

66

66


In [36]:
print(positions_counts_merged.ENSMUSG00000025917, ft.ENSMUSG00000025917, filts.ENSMUSG00000025917, sep='\n\n')

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024879, 10024880, 10024882, 10024888, 10024660, 10024882, 10024876, 10024880, 10024893, 10024899], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_barcodes=['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], read_umis=['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT', 'TCACGTGAG', 'TCCTAAATG', 'ATGATCGGT', 'TCCAGGAAT', 'TTTAGAACT', 'GCTAGGACG'], read_lengths=[46, 50, 48, 46, 43, 45, 54, 50, 41, 41], utr_sizes_from_reads=[98, 97, 95, 89, 317, 95, 101, 97, 84, 78])

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024879, 10024880, 10024882, 10024888, 10024660, 10024882, 10024876, 10024880, 10024893, 10024899], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_barcodes=['CGT

In [37]:
print(ft.ENSMUSG00000025917['read_barcodes'], filts.ENSMUSG00000025917['read_barcodes'], sep='\n\n')

['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2']

['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2']


In [38]:
%time df, peak_var, obs = peak_calling.generate_peak_counts(filts, annotations=vivo_anno)
print(df)

2021-07-12 18:06:21,042 - INFO - Generating and counting UTR size peaks
Calling peaks for each UTR: 100%|██████████| 66/66 [00:01<00:00, 37.48it/s]
CPU times: user 259 ms, sys: 185 ms, total: 443 ms
Wall time: 2.14 s
                    ENSMUSG00000025917-1  ENSMUSG00000025917-2  \
TTCTTAGGTTGGTTTG-1                   0.0                   0.0   
GCAGTTAGTAAACCTC-2                   0.0                   0.0   
AGCAGCCGTCCATCCT-2                   0.0                   0.0   
ACGATACTCGAATGGG-2                   0.0                   0.0   
TGCGCAGAGTCCATAC-1                   0.0                   0.0   
...                                  ...                   ...   
GTCGGGTTCAACACTG-1                   0.0                   0.0   
TTATGCTCAGACGCTC-1                   0.0                   0.0   
ACGATGTAGACGCTTT-2                   0.0                   0.0   
GGATTACCATGACATC-2                   0.0                   0.0   
GCTCCTATCAACACGT-2                   0.0                 

In [39]:
peak_var

,utr_size
ENSMUSG00000025917-1,78.0
ENSMUSG00000025917-2,95.0
ENSMUSG00000025917-3,317.0
ENSMUSG00000025925-1,346.0
ENSMUSG00000037509-1,904.0
...,...
ENSMUSG00000066892-1,534.0
ENSMUSG00000066839-1,18.0
ENSMUSG00000034135-1,2111.0
ENSMUSG00000031429-1,134.0


In [40]:
%time peak_var = peak_calling.add_to_peaks_var(peak_var, utr_subsetted)

2021-07-12 18:06:23,301 - INFO - Adding reference annotations to the final result
CPU times: user 8.47 ms, sys: 72 µs, total: 8.54 ms
Wall time: 8 ms


In [41]:
peak_var

,utr_size,name,chrom,start,end,score,strand,transcript_id,gene_name
ENSMUSG00000025917-1,78.0,ENSMUSG00000025917,1,10024601,10024978,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025917-2,95.0,ENSMUSG00000025917,1,10024601,10024978,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025917-3,317.0,ENSMUSG00000025917,1,10024601,10024978,0.0,-,ENSMUST00000027050,Cops5
ENSMUSG00000025925-1,346.0,ENSMUSG00000025925,1,15843083,15843470,0.0,+,ENSMUST00000027057,Terf1
ENSMUSG00000037509-1,904.0,ENSMUSG00000037509,1,34811830,34812739,0.0,+,ENSMUST00000159747,Arhgef4
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000066892-1,534.0,ENSMUSG00000066892,9,20637786,20638376,0.0,-,"ENSMUST00000086458,ENSMUST00000148631",Fbxl12
ENSMUSG00000066839-1,18.0,ENSMUSG00000066839,9,22072246,22072323,0.0,-,ENSMUST00000098937,Ecsit
ENSMUSG00000034135-1,2111.0,ENSMUSG00000034135,9,46222069,46224194,0.0,+,ENSMUST00000126865,Sik3
ENSMUSG00000031429-1,134.0,ENSMUSG00000031429,X,140948429,140949140,0.0,-,ENSMUST00000033805,Psmd10


In [42]:
count_matrix_anndata = peak_calling.generate_count_matrix_anndata(df, peak_var, obs=obs)

2021-07-12 18:06:23,370 - INFO - Generating Count Matrix


In [50]:
count_matrix_anndata.X.shape

(695, 160)

In [43]:
print(count_matrix_anndata.var)

                      utr_size                name chrom      start  \
ENSMUSG00000025917-1      78.0  ENSMUSG00000025917     1   10024601   
ENSMUSG00000025917-2      95.0  ENSMUSG00000025917     1   10024601   
ENSMUSG00000025917-3     317.0  ENSMUSG00000025917     1   10024601   
ENSMUSG00000025925-1     346.0  ENSMUSG00000025925     1   15843083   
ENSMUSG00000037509-1     904.0  ENSMUSG00000037509     1   34811830   
...                        ...                 ...   ...        ...   
ENSMUSG00000066892-1     534.0  ENSMUSG00000066892     9   20637786   
ENSMUSG00000066839-1      18.0  ENSMUSG00000066839     9   22072246   
ENSMUSG00000034135-1    2111.0  ENSMUSG00000034135     9   46222069   
ENSMUSG00000031429-1     134.0  ENSMUSG00000031429     X  140948429   
ENSMUSG00000031429-2     653.0  ENSMUSG00000031429     X  140948429   

                            end  score strand  \
ENSMUSG00000025917-1   10024978    0.0      -   
ENSMUSG00000025917-2   10024978    0.0      -   


In [48]:
count_matrix_anndata.obs

,CellType,CellAge,pTime
TTCTTAGGTTGGTTTG-1,TAP,old,27.677634
GCAGTTAGTAAACCTC-2,NB,young,32.404217
AGCAGCCGTCCATCCT-2,aNSC1,young,22.325666
ACGATACTCGAATGGG-2,aNSC2,young,21.342897
TGCGCAGAGTCCATAC-1,aNSC2,old,22.464843
...,...,...,...
GTCGGGTTCAACACTG-1,TAP,old,14.969375
TTATGCTCAGACGCTC-1,TAP,old,25.640565
ACGATGTAGACGCTTT-2,aNSC1,young,18.737798
GGATTACCATGACATC-2,TAP,young,24.214659
